In [1]:
%load_ext autoreload
%autoreload 2

from fibsem import utils, acquire
from fibsem.milling import get_milling_stages, mill_stages
from fibsem.milling.strategy import register_strategy
from fibsem.structures import BeamType
from autolamella.protocol.validation import validate_protocol
from pprint import pprint

from adaptive_polish import strategy


Default configuration default-configuration. Configuration Path: c:\Users\dmv31621\AppData\Local\miniforge3\envs\adaptivepolish\lib\site-packages\fibsem\config\microscope-configuration.yaml


c:\Users\dmv31621\AppData\Local\miniforge3\envs\adaptivepolish\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# connect to microscope
microscope, settings = utils.setup_session(
    config_path=r"C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2025\00_Adaptive-milling\fibsem\fibsem\config\microscope-configuration-demo2.yaml"
)

# load new style protocol with adaptive-polish with validation
PROTOCOL_PATH = "protocol-on-grid-ap-only.yaml"
protocol = validate_protocol(utils.load_protocol(protocol_path=PROTOCOL_PATH))

2025-03-13 10:59:22,369 — root — INFO — connect_to_microscope:5418 — Microscope client connected to DemoMicroscope with serial number 123456 and software version 0.1
2025-03-13 10:59:22,370 — root — INFO — setup_session:229 — Finished setup for session: demo_2025-03-13-10-59-22AM
2025-03-13 10:59:22,377 — root — INFO — validate_protocol:166 — Adding default milling stage for mill_rough
2025-03-13 10:59:22,377 — root — INFO — validate_protocol:166 — Adding default milling stage for microexpansion
2025-03-13 10:59:22,378 — root — INFO — validate_protocol:166 — Adding default milling stage for fiducial
2025-03-13 10:59:22,378 — root — INFO — validate_protocol:166 — Adding default milling stage for notch


In [3]:
# acquire reference images (required to register paths)
lamella_folder = r"C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2025\00_Adaptive-milling\tmp\AutoLamella-2025-03-10-22-34\01-strong-salmon"
settings.image.path = lamella_folder
acquire.take_reference_images(microscope, settings.image)

# set imaging settings folder on fib so that adaptive polish finds the correct lamella folder
fib_img_settings = microscope.get_imaging_settings(BeamType.ION)
fib_img_settings.path = lamella_folder
microscope._last_imaging_settings = fib_img_settings

milling_stages = get_milling_stages("mill_polishing", protocol["milling"])

strategy_config: strategy.AdaptivePolishMillingConfig = milling_stages[0].strategy.config
print("Adaptive Polishing Config:")
print(f"Milling Interval: {strategy_config.milling_interval_s}")
print(f"Maximum Cycles: {strategy_config.max_milling_cycles}")
print(f"Model: {strategy_config.model_path}")

2025-03-13 10:59:22,811 — root — INFO — acquire_image:6335 — acquiring new ELECTRON image.
2025-03-13 10:59:22,812 — root — INFO — acquire_image:6336 — resolution:[1536, 1024], hfw:0.00015
2025-03-13 10:59:22,814 — root — INFO — acquire_image:6343 — SEM
2025-03-13 10:59:22,815 — root — INFO — load:141 — Loading C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2024\00_Adaptive-milling\test_images\test_07-steady-gator\SEM\adapt_mill_img_000_electron_000.tif as spoof image
2025-03-13 10:59:22,951 — root — INFO — acquire_image:6370 — img_data.type:<class 'numpy.ndarray'>
2025-03-13 10:59:22,952 — root — INFO — acquire_image:6371 — img_data.dtype:uint8, img_data.shape:(2048, 3072)
2025-03-13 10:59:22,953 — root — INFO — acquire_image:6375 — demo image has shape:(2048, 3072). Resizing to (np.int64(1024), np.int64(1536))
2025-03-13 10:59:23,218 — root — INFO — acquire_image:6379 — Resized, img_data.dtype:uint8, img_data.shape:(1024, 1536)
2025-03-13 10:59:23,219 — root — INFO — ac

In [4]:
# run milling stages
mill_stages(microscope=microscope, stages=milling_stages)

2025-03-13 10:59:24,181 — root — INFO — run:100 — Running Adaptive polishing according to GIS thickness for Polishing Mill 01
2025-03-13 10:59:24,183 — root — WARNING — _set:6184 — Unknown key: active_device (BeamType.ION)
2025-03-13 10:59:24,187 — root — INFO — run:111 — Lamella folder C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2025\00_Adaptive-milling\tmp\AutoLamella-2025-03-10-22-34\01-strong-salmon/adaptive_polish already exists, some data may be overwritten.
2025-03-13 10:59:24,190 — root — INFO — __init__:174 — model path:C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2025\00_Adaptive-milling\models\2024-02-24_0013_gis_lamela_crack_pytorch_AUnet.ptchkp
2025-03-13 10:59:24,250 — root — INFO — __init__:180 — DL device: cuda
2025-03-13 10:59:25,135 — root — INFO — run:124 — Using sem beam shift alignment for adaptive polishing
2025-03-13 10:59:25,136 — root — INFO — acquire_image:6335 — acquiring new ELECTRON image.
2025-03-13 10:59:25,138 — root — IN

C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2025\00_Adaptive-milling\adaptive_polish\src\adaptive_polish\gis_measurement.py:147: RuntimeWarning: All-NaN slice encountered
  GIS_windowed = np.nanmedian(GIS_pxbypx_NaNed.reshape(-1, window_size_px), axis=1)


2025-03-13 10:59:29,147 — root — INFO — draw_rectangle:5752 — Setting pattern time to 10.
2025-03-13 10:59:30,149 — root — INFO — _handle_progress:164 — {'progress': {'state': 'update', 'start_time': 1741863569.1486778, 'milling_state': <MillingState.RUNNING: 1>, 'estimated_time': 10, 'remaining_time': 9}}
2025-03-13 10:59:31,158 — root — INFO — _handle_progress:164 — {'progress': {'state': 'update', 'start_time': 1741863569.1486778, 'milling_state': <MillingState.RUNNING: 1>, 'estimated_time': 10, 'remaining_time': 8}}
2025-03-13 10:59:32,166 — root — INFO — _handle_progress:164 — {'progress': {'state': 'update', 'start_time': 1741863569.1486778, 'milling_state': <MillingState.RUNNING: 1>, 'estimated_time': 10, 'remaining_time': 7}}
2025-03-13 10:59:33,178 — root — INFO — _handle_progress:164 — {'progress': {'state': 'update', 'start_time': 1741863569.1486778, 'milling_state': <MillingState.RUNNING: 1>, 'estimated_time': 10, 'remaining_time': 6}}
2025-03-13 10:59:34,184 — root — INFO 

C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2025\00_Adaptive-milling\adaptive_polish\src\adaptive_polish\gis_measurement.py:147: RuntimeWarning: All-NaN slice encountered
  GIS_windowed = np.nanmedian(GIS_pxbypx_NaNed.reshape(-1, window_size_px), axis=1)


2025-03-13 10:59:41,354 — root — INFO — run:263 — Stopping as crack area (um2) 2.0980834960937496 > threshold 2 um2
2025-03-13 10:59:41,554 — root — INFO — finish_milling:86 — Changing to Imaging Current: 2.00e-11
2025-03-13 10:59:41,557 — root — INFO — finish_milling:5724 — Finishing milling: 2.00e-11
2025-03-13 10:59:41,558 — root — INFO — finish_milling:88 — Finished Ion Beam Milling.
2025-03-13 10:59:41,559 — root — INFO — finish_milling:86 — Changing to Imaging Current: 2.00e-11
2025-03-13 10:59:41,560 — root — INFO — finish_milling:5724 — Finishing milling: 2.00e-11
2025-03-13 10:59:41,562 — root — INFO — finish_milling:88 — Finished Ion Beam Milling.


In [5]:
# for more detailed error finding
# milling_stages[0].strategy.run(microscope, milling_stages[0])